# GPT-2 XL: Factual Recall Baseline

48 layers, d_model=1600, d_mlp=6400, 25 heads.
Largest GPT-2 model. Same prompts as smaller models.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [2]:
model, info = load_model("gpt2-xl")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-xl into HookedTransformer
Loaded gpt2-xl on mps
  48 layers | 25 heads | d_model=1600 | d_mlp=6400 | sequential attn+MLP


In [3]:
prompts = [
    ("The capital of France is", " Paris", "paris"),
    ("The capital of Germany is", " Berlin", "berlin"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  PARIS: "The capital of France is" → " Paris"

Model: gpt2-xl
Prompt: "The capital of France is"
Target: " Paris" (token 6342)
Final prediction: " a"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         now                 2624           0.000065    
1         not                 1952           0.000090    
2         not                 1921           0.000090    
3         not                 1871           0.000091    
4         not                 1953           0.000089    
5         currently           1199           0.000134    
6         currently           602            0.000222    
7         believed            901            0.000164    
8         believed            1661           0.000087    
9         slated              1757           0.000084    
10        slated              1417           0.000102    
11        poised              947            0.000157    
12        poised              408     

In [ ]:
for label, data in results.items():
    print(f"\n--- {label} ---")
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [ ]:
model_name = "gpt2-xl"

for label, data in results.items():
    data["lens"].to_csv(f"../../results/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/{model_name}")

In [6]:
unload(model)

Model unloaded, memory cleared
